# The Power Hour: Forecasting Electricity Demand

XGBoost regressor predicting hourly electricity demand for the PJM East region from engineered time series features.

**Dataset:** [Hourly Energy Consumption](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption) - 145,366 hourly observations from PJM East Interconnection (2002-2018).

**Approach:** Temporal feature engineering (calendar, cyclical, lags, rolling stats) + XGBoost with hyperparameter tuning and SHAP interpretability.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import ParameterSampler
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 30)

In [ ]:
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
}

plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def fmt_k(x, _=None):
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
df = pd.read_csv("data/PJME_hourly.csv")
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)
df = df.set_index("Datetime")

# Handle DST: drop duplicate timestamps, forward-fill single-hour gaps
df = df[~df.index.duplicated(keep="first")]
df = df.asfreq("h")
df["PJME_MW"] = df["PJME_MW"].ffill()

print(f"Loaded: {len(df):,} hourly observations")
print(f"Date range: {df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d}")
print(f"Frequency: {df.index.freq}")
print(f"Missing values: {df['PJME_MW'].isna().sum()}")
print(f"\nDemand stats (MW):")
print(df["PJME_MW"].describe().round(0))

---
## Exploratory Data Analysis

In [ ]:
daily = df["PJME_MW"].resample("D").mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily.index, daily.values, lw=0.5, color=C["accent"], alpha=0.6)
rolling_30 = daily.rolling(30).mean()
ax.plot(rolling_30.index, rolling_30.values, lw=2, color=C["primary"], label="30-day rolling mean")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="PJM East Daily Average Demand (2002-2018)",
           subtitle="16 years of hourly electricity consumption",
           xlabel="", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_plot = df.copy()
df_plot["month"] = df_plot.index.month
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bp = df_plot.boxplot(column="PJME_MW", by="month", ax=ax, patch_artist=True,
                      boxprops=dict(facecolor=C["accent"], alpha=0.3),
                      medianprops=dict(color=C["risk"], lw=2),
                      flierprops=dict(marker=".", markersize=1, alpha=0.05),
                      showfliers=True)
ax.set_xticklabels(month_names)
ax.set_title("")
plt.suptitle("")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
style_axis(ax, title="Hourly Demand Distribution by Month",
           subtitle="Summer AC and winter heating drive the highest peaks",
           xlabel="Month", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
df_plot["hour"] = df_plot.index.hour
df_plot["is_weekend"] = df_plot.index.dayofweek >= 5

hourly_weekday = df_plot[~df_plot["is_weekend"]].groupby("hour")["PJME_MW"].mean()
hourly_weekend = df_plot[df_plot["is_weekend"]].groupby("hour")["PJME_MW"].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly_weekday.index, hourly_weekday.values, lw=2.5, color=C["accent"],
        marker="o", markersize=4, label="Weekday")
ax.plot(hourly_weekend.index, hourly_weekend.values, lw=2.5, color=C["warn"],
        marker="s", markersize=4, label="Weekend")
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Average Hourly Load Curve: Weekday vs Weekend",
           subtitle="Weekdays peak during business hours, weekends are flatter and lower",
           xlabel="Hour of Day", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_demand = df.groupby(df.index.dayofweek)["PJME_MW"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors_dow = [C["accent"]] * 5 + [C["warn"]] * 2
ax.bar(range(7), dow_demand.values, color=colors_dow, alpha=0.8)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
for i, v in enumerate(dow_demand.values):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
style_axis(ax, title="Average Demand by Day of Week",
           subtitle="Amber = weekend. Clear weekday/weekend separation.",
           xlabel="", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = df.copy()
heatmap_data["hour"] = heatmap_data.index.hour
heatmap_data["month"] = heatmap_data.index.month
pivot = heatmap_data.groupby(["hour", "month"])["PJME_MW"].mean().unstack()
pivot.columns = month_names

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, linewidths=0.3, linecolor="#e2e8f0",
            fmt=",.0f", annot=True, annot_kws={"size": 8},
            cbar_kws={"label": "Avg Demand (MW)"})
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
style_axis(ax, title="Average Demand Heatmap: Hour x Month",
           subtitle="Peak demand at summer afternoons (July-Aug, 14:00-18:00) and winter mornings",
           xlabel="Month", ylabel="Hour of Day")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
compare_years = [2005, 2010, 2015]
colors_yoy = [C["accent"], C["safe"], C["warn"]]

for year, color in zip(compare_years, colors_yoy):
    year_data = daily[daily.index.year == year]
    ax.plot(range(len(year_data)), year_data.values, lw=1.5, color=color,
            alpha=0.8, label=str(year))

ax.set_xlabel("Day of Year")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Year-over-Year Demand Comparison",
           subtitle="Daily average demand for 2005, 2010, 2015 - consistent seasonal shape",
           xlabel="Day of Year", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

---
## Feature Engineering

All features derived from the datetime index and lagged values of the target. No external data sources - everything the model needs comes from the timestamp and demand history.

In [ ]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["dayofyear"] = df.index.dayofyear
df["weekofyear"] = df.index.isocalendar().week.astype(int)
df["quarter"] = df.index.quarter
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

# Cyclical encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

print(f"Calendar + cyclical features added: {df.shape[1] - 1} features")

In [ ]:
df["lag_1h"] = df["PJME_MW"].shift(1)
df["lag_24h"] = df["PJME_MW"].shift(24)
df["lag_48h"] = df["PJME_MW"].shift(48)
df["lag_168h"] = df["PJME_MW"].shift(168)

print("Lag features added: lag_1h, lag_24h, lag_48h, lag_168h")

In [ ]:
df["rolling_24h_mean"] = df["PJME_MW"].shift(1).rolling(24).mean()
df["rolling_24h_std"] = df["PJME_MW"].shift(1).rolling(24).std()
df["rolling_7d_mean"] = df["PJME_MW"].shift(1).rolling(168).mean()
df["rolling_7d_std"] = df["PJME_MW"].shift(1).rolling(168).std()

print("Rolling features added: 24h mean/std, 7d mean/std")

In [ ]:
df["days_since_start"] = (df.index - df.index.min()).days

print(f"\nTotal features: {df.shape[1] - 1}")
print(f"NaN rows (from lags/rolling): {df.isna().any(axis=1).sum():,}")
print(f"\nFeature list:")
print([c for c in df.columns if c != "PJME_MW"])

### Feature Matrix Preview

In [ ]:
print(df.dropna().head(10).to_string())

---
## Train / Validation / Test Split

Temporal split - no data leakage across time:
- **Train:** 2002 through 2016-07-31
- **Validation:** 2016-08-01 through 2017-07-31
- **Test:** 2017-08-01 through 2018-08-03

In [ ]:
df_model = df.dropna().copy()

target = "PJME_MW"
feature_cols = [c for c in df_model.columns if c != target]

train = df_model[df_model.index < "2016-08-01"]
val = df_model[(df_model.index >= "2016-08-01") & (df_model.index < "2017-08-01")]
test = df_model[df_model.index >= "2017-08-01"]

X_train, y_train = train[feature_cols], train[target]
X_val, y_val = val[feature_cols], val[target]
X_test, y_test = test[feature_cols], test[target]

print(f"Train: {len(X_train):,} obs  ({X_train.index.min():%Y-%m-%d} to {X_train.index.max():%Y-%m-%d})")
print(f"Val:   {len(X_val):,} obs  ({X_val.index.min():%Y-%m-%d} to {X_val.index.max():%Y-%m-%d})")
print(f"Test:  {len(X_test):,} obs  ({X_test.index.min():%Y-%m-%d} to {X_test.index.max():%Y-%m-%d})")
print(f"Features: {len(feature_cols)}")

---
## XGBoost Model

### Hyperparameter Tuning

Randomized search over a structured parameter grid, scored on validation MAPE. Low learning rates with generous tree budgets and early stopping.

In [ ]:
param_grid = {
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 0.9],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [1.0, 2.0, 5.0],
    "learning_rate": [0.005, 0.01, 0.02],
}

INT_PARAMS = {"max_depth", "min_child_weight"}

n_iter = 30
sampled_params = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))

results = []
for i, params in enumerate(sampled_params):
    params = {k: int(v) if k in INT_PARAMS else v for k, v in params.items()}

    model_i = xgb.XGBRegressor(
        n_estimators=3000,
        eval_metric="mape",
        early_stopping_rounds=50,
        random_state=42,
        **params,
    )
    model_i.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=0)

    y_pred_val = model_i.predict(X_val)
    val_mape = mean_absolute_percentage_error(y_val, y_pred_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

    results.append({
        "iter": i,
        "val_mape": val_mape,
        "val_rmse": val_rmse,
        "best_iteration": model_i.best_iteration,
        **params,
    })
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{n_iter} done - best MAPE so far: {min(r['val_mape'] for r in results):.4f}")

tuning_df = pd.DataFrame(results).sort_values("val_mape", ascending=True)
print(f"\nTop 5 configurations by validation MAPE:")
print(tuning_df.head()[["val_mape", "val_rmse", "best_iteration",
                         "learning_rate", "max_depth", "min_child_weight",
                         "subsample", "colsample_bytree", "gamma",
                         "reg_alpha", "reg_lambda"]].to_string(index=False))

### Tuning Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, param in zip(axes.flatten(), ["learning_rate", "max_depth", "min_child_weight",
                                       "subsample", "colsample_bytree", "gamma"]):
    ax.scatter(tuning_df[param], tuning_df["val_mape"], alpha=0.6, s=40, color=C["accent"])
    ax.set_xlabel(param)
    ax.set_ylabel("Val MAPE")
    ax.set_title(f"MAPE vs {param}")

plt.suptitle("Hyperparameter Search Results (30 iterations)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"\nMAPE range: {tuning_df['val_mape'].min():.4f} - {tuning_df['val_mape'].max():.4f}")

### Best Model - Retrain

In [ ]:
best_params = tuning_df.iloc[0].to_dict()
best_params = {k: int(v) if k in INT_PARAMS else v
               for k, v in best_params.items() if k in param_grid.keys()}

print("Best hyperparameters:")
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

model = xgb.XGBRegressor(
    n_estimators=5000,
    eval_metric="mape",
    early_stopping_rounds=100,
    random_state=42,
    **best_params,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)
print(f"\nBest iteration: {model.best_iteration}")

---
## Model Evaluation

In [ ]:
y_pred_test = model.predict(X_test)
y_pred_val = model.predict(X_val)

test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae = mean_absolute_error(y_test, y_pred_test)

val_mape = mean_absolute_percentage_error(y_val, y_pred_val)
val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
val_mae = mean_absolute_error(y_val, y_pred_val)

print("         MAPE      RMSE       MAE")
print(f"Val:   {val_mape:.4f}   {val_rmse:,.0f} MW   {val_mae:,.0f} MW")
print(f"Test:  {test_mape:.4f}   {test_rmse:,.0f} MW   {test_mae:,.0f} MW")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(y_test.index, y_test.values, lw=0.5, color=C["accent"], alpha=0.6, label="Actual")
ax.plot(y_test.index, y_pred_test, lw=0.5, color=C["risk"], alpha=0.6, label="Predicted")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Forecast vs Actual - Full Test Period",
           subtitle=f"MAPE = {test_mape:.2%} | RMSE = {test_rmse:,.0f} MW",
           xlabel="", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

summer_start, summer_end = "2018-07-09", "2018-07-22"
mask_s = (y_test.index >= summer_start) & (y_test.index <= summer_end)
ax = axes[0]
ax.plot(y_test.index[mask_s], y_test.values[mask_s], lw=2, color=C["accent"], label="Actual")
ax.plot(y_test.index[mask_s], y_pred_test[mask_s], lw=2, color=C["risk"], ls="--", label="Predicted")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Summer Peak Week (Jul 9-22, 2018)",
           subtitle="Can the model track daily peaks during high-demand summer days?",
           ylabel="Demand (MW)")

winter_start, winter_end = "2018-01-08", "2018-01-21"
mask_w = (y_test.index >= winter_start) & (y_test.index <= winter_end)
ax = axes[1]
ax.plot(y_test.index[mask_w], y_test.values[mask_w], lw=2, color=C["accent"], label="Actual")
ax.plot(y_test.index[mask_w], y_pred_test[mask_w], lw=2, color=C["risk"], ls="--", label="Predicted")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Winter Week (Jan 8-21, 2018)",
           subtitle="Winter heating demand with different daily profile than summer",
           ylabel="Demand (MW)")

plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test.values - y_pred_test

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(residuals, bins=80, color=C["accent"], alpha=0.5, density=True, edgecolor="none")
ax.axvline(0, color=C["primary"], lw=2, ls="--")
ax.axvline(np.mean(residuals), color=C["risk"], lw=1.5, ls=":",
           label=f"Mean: {np.mean(residuals):,.0f} MW")
ax.axvline(np.median(residuals), color=C["safe"], lw=1.5, ls=":",
           label=f"Median: {np.median(residuals):,.0f} MW")
ax.legend()
style_axis(ax, title="Residual Distribution (Actual - Predicted)",
           subtitle=f"Std dev: {np.std(residuals):,.0f} MW. Centered near zero = no systematic bias.",
           xlabel="Residual (MW)", ylabel="Density")
plt.tight_layout()
plt.show()

In [ ]:
error_by_hour = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred_test,
    "abs_error": np.abs(residuals),
    "hour": y_test.index.hour,
})
hourly_mae = error_by_hour.groupby("hour")["abs_error"].mean()

fig, ax = plt.subplots(figsize=(12, 5))
colors_hour = [C["risk"] if v > hourly_mae.median() else C["accent"] for v in hourly_mae.values]
ax.bar(hourly_mae.index, hourly_mae.values, color=colors_hour, alpha=0.8)
ax.axhline(hourly_mae.median(), color=C["dark_gray"], ls="--", lw=1,
           label=f"Median MAE: {hourly_mae.median():,.0f} MW")
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Mean Absolute Error by Hour of Day",
           subtitle="Red = above-median error. Peak hours are hardest to predict.",
           xlabel="Hour of Day", ylabel="MAE (MW)")
plt.tight_layout()
plt.show()

In [ ]:
error_by_month = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred_test,
    "pct_error": np.abs(residuals) / y_test.values,
    "month": y_test.index.month,
})
monthly_mape = error_by_month.groupby("month")["pct_error"].mean()
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 5))
colors_month = [C["risk"] if v > monthly_mape.median() else C["accent"] for v in monthly_mape.values]
ax.bar(monthly_mape.index, monthly_mape.values, color=colors_month, alpha=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels([month_labels[m-1] for m in monthly_mape.index])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.axhline(monthly_mape.median(), color=C["dark_gray"], ls="--", lw=1,
           label=f"Median MAPE: {monthly_mape.median():.2%}")
ax.legend()
style_axis(ax, title="MAPE by Month",
           subtitle="Red = above-median error. Summer peaks and seasonal transitions are hardest.",
           xlabel="Month", ylabel="MAPE")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test.values, y_pred_test, s=2, alpha=0.1, color=C["accent"])
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, "k--", lw=1, alpha=0.5, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Actual vs Predicted Demand",
           subtitle="Points on the diagonal = perfect. Spread = error.",
           xlabel="Actual (MW)", ylabel="Predicted (MW)")
plt.tight_layout()
plt.show()

---
## Feature Importance (SHAP)

Which temporal patterns does the model rely on most? SHAP values decompose each prediction into feature-level contributions.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title("SHAP Feature Importance - Top 15", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("Mean |SHAP Value| - Top 15 Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Feature Importance Interpretation

The SHAP analysis reveals which temporal signals drive the model's predictions. Key takeaways:

- **Lag features dominate.** The most recent demand readings (lag_1h, lag_24h, lag_168h) carry the most predictive weight. Electricity demand is highly autocorrelated.
- **Rolling means matter.** The 24-hour and 7-day rolling averages capture the current demand regime - is it a high-demand week or a mild one?
- **Calendar features add structure.** Hour of day and day of week encode the daily and weekly cycles that lag features alone cannot capture cleanly.
- **Cyclical encoding works.** The sin/cos pairs show up as meaningful features, confirming the model benefits from smooth periodic representations over raw integer encodings.
- **Trend is minor.** `days_since_start` has low importance, suggesting demand patterns are more seasonal than trend-driven over this period.